In [0]:
import os, sys

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from src.data import load_cmapss, ALL_COLS
print(len(ALL_COLS), ALL_COLS[:6])

In [0]:
CATALOG = "turbofan"   # change to your catalog name from T04
SCHEMA  = "cmapss"
VOL     = f"/Volumes/{CATALOG}/{SCHEMA}/raw"

In [0]:
import pandas as pd

train = load_cmapss(f"{VOL}/train_FD001.txt")
test  = load_cmapss(f"{VOL}/test_FD001.txt")

rul = pd.read_csv(f"{VOL}/RUL_FD001.txt", sep=r"\s+", header=None, names=["rul_true"])
rul.insert(0, "unit", range(1, len(rul) + 1))   # row i = test engine i

for df, name in [(train, "train_FD001.txt"), (test, "test_FD001.txt"), (rul, "RUL_FD001.txt")]:
    df["source_file"] = name

print(train.shape, test.shape, rul.shape)

In [0]:
tables = {
    "bronze_train_fd001": train,
    "bronze_test_fd001":  test,
    "bronze_rul_fd001":   rul,
}
for name, pdf in tables.items():
    (spark.createDataFrame(pdf)
          .write.mode("overwrite")
          .saveAsTable(f"{CATALOG}.{SCHEMA}.{name}"))
    print("written", name)

In [0]:
for name in tables:
    t = spark.table(f"{CATALOG}.{SCHEMA}.{name}")
    print(name, "rows:", t.count(), "engines:", t.select("unit").distinct().count(), "cols:", len(t.columns))